In [ ]:
### Llamm_index
    - Embedding
    - Retriever             : index.as_retriever()
    - Query Engine          : index.as_query_engine()
    - Vector DB Management  : index.insert(doc), index.update_ref_doc(doc), index.refresh_ref_docs([]), index.delete_ref_doc(doc_id)

    - VectorStoreindex, SimpleDirectoryReader, Document, PromptTemplate, SentenceSplitter 주요

In [ ]:
### 1. imports

import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Document, PromptTemplate
from openai import OpenAI

from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.retrievers import BaseRetriever
from llama_index.core import get_response_synthesizer
from llama_index.core.response_synthesizers import BaseSynthesizer

import textwrap
import openai

In [ ]:
### 2. Building a Query Engine

## 2-1. load document from directory
documents = SimpleDirectoryReader("data").load_data()
print(len(documents))

for i in range(len(documents)):
  print(documents[i].text)


## 2-2. document -> nodes (embedding)
index = VectorStoreIndex.from_documents( documents )

node_id = index.index_struct.nodes_dict

for key, value in node_id.items():
    print(key, value)
    node_example_id = value
    break

print("The number of nodes: ", len(node_id.values()))           # 21 nodes
print(node_id.values())


## 2-3. check node vector dimension
v = index.vector_store.data.embedding_dict[ node_example_id ]
print(len(v))                                                   # 1536 dimension


## 2-4. get Query Engine from index
query_engine = index.as_query_engine()

In [ ]:
### 3. Default options of Llama_index options
    - parser : SentenceSplitter( chunk_size=1024, chunk_overlap=200 )
    - tokenizer : tiktoken.encoding_for_model( "gpt-3.5-turbo" )

## 3.1 using SentenceSplitter
from llama_index.core.node_parser import SentenceSplitter

parser = SentenceSplitter( chunk_size=1024, chunk_overlap=200 )     # default
nodes = parser.get_nodes_from_documents( documents )

print(len(nodes))                                                   # 21


## 3.2 using embedding
import tiktoken
tokenizer = tiktoken.encoding_for_model( "gpt-3.5-turbo" )

encoded_text = tokenizer.encode(nodes[0].text)
print(f"Encoded text: {encoded_text}")
print(f"Encoded text length: {len(encoded_text)}")

decoded_text = tokenizer.decode(encoded_text)
print(f"Decoded text: \n\n{decoded_text}")


## 3.3 apply text splitter to make index
text_splitter = SentenceSplitter( chunk_size=200, chunk_overlap=50 )
index = VectorStoreIndex.from_documents( documents=documents, transformations=[text_splitter] )

In [ ]:
### 4. Answer for the questions
    - query_engine.query()

## 4.1 Anwsering
response = query_engine.query( "What is the first programs the author tried writing?" )
print(response)


## 4.2 utility function to get response of ChatGPT result
def generate_answer(question):
    messages = [
        {
            "role": "user",
            "content": question,
        },
    ]
    
    response = openai.chat.completions.create(
        model="gpt-3.5-turbo",
        temperature=0,
        messages=messages,
    )

    return response.choices[0].message.content

In [ ]:
### 5. Retriever from index
    - retriever = index.as_retriever()
    - ret_passages = retriever.retrieve( "Who is the author?" )

retriever = index.as_retriever()

ret_passages = retriever.retrieve("Who is the author?")

for i in range(len(ret_passages)):
  print("###Retrieved Passage\n", ret_passages[i].text)
  print("\n\n\n")

In [ ]:
### 6. Vector DB Management

## 6.1 insert new doc

docu = Document( text=data_text, id_="new_doc_id" )
index.insert( docu )

new_query_engine = index.as_query_engine()


## 6.2 update doc

docu.set_content( value="Natural diamonds were (and are) formed (thousands of million years ago) in the upper mantle of Earth in metallic melts at temperatures of 2,000–6,000 °C and at pressures of 8–9 GPa." )
output = index.update_ref_doc(
    docu,
    update_kwargs={"delete_kwargs": {"delete_from_docstore": True}},
)

query_engine_update = index.as_query_engine()
res_update = query_engine_update.query(question)


## 6.3 update doc list
docu.set_content( value="Natural diamonds were (and are) formed (thousands of million years ago) in the upper mantle of Earth in metallic melts at temperatures of 6,000–8,000 °C and at pressures of 12–15 GPa." )
output = index.refresh_ref_docs( [docu] )

query_engine_refresh = index.as_query_engine()
res_refresh = query_engine_refresh.query(question)


## 6.4 delete doc

id = docu.doc_id
index.delete_ref_doc( id, delete_from_docstore=True )

query_engine_delete = index.as_query_engine()
res_delete = query_engine_delete.query(question)

print(index.ref_doc_info.keys())
